# CPT Notebook B — Train (Section 6.1)

**Chạy lại notebook này mỗi session Kaggle (Run All), không sửa gì giữa các session.** Mỗi lần chạy nó tự: kéo checkpoint mới nhất từ Hub → train tiếp → push checkpoint đầy đủ mỗi `save_steps` → tự dừng trước khi hết giờ session. Lặp đến khi `global_step >= 3000` thì cell cuối tự merge LoRA và push bản final.

3 repo trên Hub (đừng nhầm):
- `{HF_USER}/vi-cpt-corpus-2048` — data đã pack (từ Notebook A, chạy trước 1 lần)
- `{HF_USER}/Qwen3-1.7B-vi-cpt-ckpt` — checkpoint để resume (adapter + optimizer + scheduler)
- `{HF_USER}/Qwen3-1.7B-vi-cpt` — bản merged cuối cùng, input cho Bước 2 (SFT)

**Trước khi chạy:** bật GPU T4; vào **Add-ons → Secrets** và **TICK cả `HF_TOKEN` (Write token) lẫn `WANDB_API_KEY`** cho đúng notebook này — secret attach theo từng notebook, import notebook mới là phải tick lại. Username HF tự lấy từ token, không cần sửa gì trong code.

**Xem log khi chạy batch (Save & Run All):** mở notebook đang chạy → panel bên phải hoặc **⋮ → View Active Events → Logs**. Trang notebook tĩnh chỉ hiện output sau khi run KẾT THÚC — không thấy gì ở đó không có nghĩa là treo.

**Mốc thời gian mỗi session** (đo thực tế ~95-106 s/step, ~35-38 step/giờ): dòng loss đầu tiên ~2-3 phút sau khi vào train, heartbeat + loss mỗi ~35 phút (`logging_steps=20`); checkpoint push lên Hub mỗi 200 step ≈ mỗi ~5h30-5h50; **eval chạy mỗi 620 step** (cố ý lệch mốc save để eval có chết cũng không kéo theo checkpoint chưa kịp lưu; batch eval=2 nên mỗi lần eval mất ~20-30 phút — heartbeat đứng yên trong lúc đó là bình thường); hết `BUDGET_H` giờ thì tự save ngay tại step đang đứng rồi dừng, một session đi được ~350 step.

In [ ]:
!pip install -q -U unsloth

In [ ]:
# Config + login — username TỰ LẤY từ token (whoami), không gõ tay để khỏi sai namespace
import os, time
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # chống phân mảnh VRAM — phải đặt TRƯỚC khi import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download, whoami
login(UserSecretsClient().get_secret("HF_TOKEN"))

info = whoami()
HF_USER = info["name"]
role = ((info.get("auth") or {}).get("accessToken") or {}).get("role")
assert role != "read", "HF_TOKEN là READ token — tạo token WRITE tại hf.co/settings/tokens rồi cập nhật Kaggle Secret"
print(f"HF user: {HF_USER} (token: {role})")

# W&B (tùy chọn): thêm secret WANDB_API_KEY (lấy tại wandb.ai/authorize) là tự bật dashboard;
# không có secret thì train vẫn chạy bình thường, log nằm trong trainer_state.json của checkpoint
try:
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_PROJECT"] = "vi-chatbot-rlhf"
    REPORT_TO = "wandb"
    print("W&B: BẬT — xem dashboard tại wandb.ai, project vi-chatbot-rlhf")
except Exception:
    REPORT_TO = "none"
    print("*** W&B: TẮT — secret WANDB_API_KEY không đọc được. Nếu bạn ĐÃ tạo secret mà vẫn thấy "
          "dòng này: secret chưa được TICK cho notebook này (Add-ons → Secrets → tick checkbox). "
          "Train vẫn chạy bình thường, log loss vẫn in ra Logs bên dưới. ***")

DATA_REPO  = f"{HF_USER}/vi-cpt-corpus-2048"
CKPT_REPO  = f"{HF_USER}/Qwen3-1.7B-vi-cpt-ckpt"   # checkpoint resume
FINAL_REPO = f"{HF_USER}/Qwen3-1.7B-vi-cpt"        # CHỈ push khi đạt MAX_STEPS
OUT_DIR    = "/kaggle/working/cpt-checkpoints"
MAX_STEPS  = 3000    # ~400M token: effective batch 64 (4 x 2 GPU x 8 accum) = 131k token/step; đo ~95-106 s/step trên T4 → ~80h GPU
BUDGET_H   = 10.0    # tính từ lúc load model; session batch sống ~11.5-12h (đo thực tế), setup ~0.5h → dừng ở ~10.5h là còn dư an toàn

In [ ]:
# Kéo checkpoint mới nhất từ Hub về (None nếu là session đầu tiên)
api = HfApi()
api.create_repo(CKPT_REPO, private=True, exist_ok=True)
steps = sorted(int(f.split("/")[0].split("-")[1]) for f in api.list_repo_files(CKPT_REPO)
               if f.startswith("checkpoint-"))
last_ckpt = None
if steps:
    name = f"checkpoint-{steps[-1]}"
    snapshot_download(CKPT_REPO, allow_patterns=f"{name}/*", local_dir=OUT_DIR)
    last_ckpt = os.path.join(OUT_DIR, name)
print("Resume từ:", last_ckpt or "(bắt đầu mới)")

In [ ]:
# Base model 4-bit + QLoRA r=64 (cấu hình cố định của Bước 1 — xem bảng 6.1 trong spec)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3-1.7B-Base", max_seq_length=2048, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=64, lora_alpha=64, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","up_proj","down_proj","o_proj","gate_proj"],
    use_rslora=True, use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
# Callback: mỗi lần save, upload NGUYÊN thư mục checkpoint lên Hub
# (phải là cả folder — optimizer.pt/scheduler.pt/rng_state/trainer_state.json — thì session sau
#  mới resume đúng; push_to_hub=True của TrainingArguments chỉ đẩy adapter nên KHÔNG dùng).
# Ngân sách giờ kiểm tra ở on_step_end: đến giờ là ÉP save ngay tại step hiện tại rồi dừng —
# không đợi mốc save_steps=200 (200 step ≈ 5.7h; đợi mốc thì có thể bị Kaggle cắt trước khi kịp save,
# session 84c514f đã vượt budget ở ~step 280 mà phải chạy tới step 400/11.5h mới dừng được).
# Hai lớp log, đều in TEXT ra stdout (bảng HTML widget của transformers KHÔNG hiện trong tab Logs
# khi chạy batch — Save & Run All — nên nhìn như treo dù đang train):
#  - on_step_end: [heartbeat] mỗi 20 step — độc lập với trainer.log, chắc chắn nổ nếu loop còn chạy,
#    ghi thẳng perf/sec_per_step lên W&B (không qua WandbCallback)
#  - on_log: loss/lr thật — nếu thấy heartbeat mà KHÔNG thấy dòng loss = trainer không bắn on_log
#    (bug Unsloth × transformers 5.x) → báo lại để vá tiếp
from transformers import TrainerCallback
class KaggleSessionCallback(TrainerCallback):
    def __init__(self):
        self.t0 = time.time(); self._last = (0, self.t0)
    def on_train_begin(self, args, state, control, **kw):
        self._last = (state.global_step, time.time())  # khi resume, global_step > 0 — không reset thì heartbeat đầu tính s/step sai
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == 1 or state.global_step % 20 == 0:
            now = time.time()
            spd = (now - self._last[1]) / max(state.global_step - self._last[0], 1)
            self._last = (state.global_step, now)
            print(f"[heartbeat] step {state.global_step}/{state.max_steps}, {spd:.0f} s/step, "
                  f"{(now - self.t0)/3600:.2f}h", flush=True)
            if REPORT_TO == "wandb":
                import wandb
                if wandb.run: wandb.log({"perf/sec_per_step": spd}, step=state.global_step)
        if time.time() - self.t0 > BUDGET_H * 3600 and not control.should_training_stop:
            control.should_save = True
            control.should_training_stop = True
            print(f"Hết ngân sách {BUDGET_H}h — save ngay tại step {state.global_step} rồi dừng, "
                  "session sau tự resume.", flush=True)
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            msg = ", ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}" for k, v in logs.items())
            print(f"[step {state.global_step}/{state.max_steps} | {(time.time()-self.t0)/3600:.2f}h] {msg}", flush=True)
    def on_save(self, args, state, control, **kw):
        name = f"checkpoint-{state.global_step}"
        api.upload_folder(repo_id=CKPT_REPO, folder_path=os.path.join(args.output_dir, name),
                          path_in_repo=name)
        print(f"Đã push {name} lên {CKPT_REPO}", flush=True)

In [ ]:
# Train — data đã pack sẵn từ Notebook A, không tokenize lại
# fp16/bf16 tự chọn theo GPU: T4 (Turing) KHÔNG có bf16 → phải fp16; A100/L4 thì bf16
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported
data = load_dataset(DATA_REPO)

# W&B: TỰ init trước, timeout 300s — KHÔNG để WandbCallback tự init (mặc định timeout 90s,
# không retry, và CommError của nó ném ngay trong on_train_begin làm CHẾT CẢ TRAIN — đã dính
# 1 session 2026-07-19). W&B lỗi thì hạ xuống "none" và train tiếp: dashboard là tiện ích,
# không bao giờ được phép giết training.
RUN_NAME = f"cpt-{time.strftime('%m%d-%H%M')}"
if REPORT_TO == "wandb":
    try:
        import wandb
        wandb.init(project=os.environ["WANDB_PROJECT"], name=RUN_NAME,
                   settings=wandb.Settings(init_timeout=300))
        print(f"W&B run: {wandb.run.url}", flush=True)
    except Exception as e:
        REPORT_TO = "none"
        print(f"*** W&B: init lỗi ({type(e).__name__}: {e}) — TẮT W&B, train vẫn chạy, "
              "log loss vẫn in ra Logs bên dưới. ***", flush=True)

trainer = Trainer(
    model=model,
    train_dataset=data["train"],
    eval_dataset={"vi": data["eval_vi"], "en": data["eval_en"]},  # eval_loss en tăng mạnh = forgetting
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[KaggleSessionCallback()],
    args=TrainingArguments(
        output_dir=OUT_DIR,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,
        learning_rate=1.5e-5, lr_scheduler_type="cosine", warmup_steps=300,  # = 10% của 3000 step
        max_steps=MAX_STEPS,
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        optim="paged_adamw_8bit",
        average_tokens_across_devices=False,  # transformers 5.x mặc định True → vỡ fused loss của Unsloth ('int' has no .mean, issue #3695)
        save_strategy="steps", save_steps=200, save_total_limit=2,
        # Eval: lúc train Unsloth dùng fused loss (không bao giờ materialize logits) nhưng lúc EVAL thì
        # model trả nguyên logits [batch, 2048, vocab 151k] rồi accelerate ép lên fp32 — batch eval mặc
        # định (8, Unsloth nhân đôi thành 16) cần 18.5 GiB → OOM chết session ở step 600 (2026-07-19,
        # mất 200 step vì transformers chạy eval TRƯỚC save cùng mốc). batch 2 → logits fp32 ~2.5 GiB.
        per_device_eval_batch_size=2, prediction_loss_only=True,
        # eval lệch mốc save (620 thay vì 600): save-200 luôn xong trước, eval có chết vì bất cứ gì
        # thì session sau cũng chỉ mất ≤ 20 step thay vì 200
        eval_strategy="steps", eval_steps=620,
        logging_steps=20, logging_first_step=True,  # log ngay step 1 — chạy 2-3 phút là biết sống hay chết
        report_to=REPORT_TO, run_name=RUN_NAME,  # WandbCallback thấy wandb.run có sẵn thì dùng lại, không init nữa
    ),
)
trainer.train(resume_from_checkpoint=last_ckpt)  # None -> train mới; có path -> nối tiếp đúng step
print(f"global_step = {trainer.state.global_step} / {MAX_STEPS}")

In [ ]:
# Khi đạt MAX_STEPS: merge LoRA vào base, push bản final — Bước 2 (SFT) load thẳng repo này
if trainer.state.global_step >= MAX_STEPS:
    model.push_to_hub_merged(FINAL_REPO, tokenizer, save_method="merged_16bit")
    print(f"CPT HOÀN TẤT — bản merged tại {FINAL_REPO}. Cập nhật README (bước 1 → done).")
else:
    print("Chưa đạt MAX_STEPS — mở lại notebook này ở session sau, nó tự resume.")

In [ ]:
# Smoke test nhanh (chỉ có ý nghĩa khi đã train kha khá / hoàn tất):
# model nên sinh tiếng Việt mạch lạc hơn base gốc — xem thêm tiêu chí 6.9 trong spec
FastLanguageModel.for_inference(model)
inputs = tokenizer("Việt Nam là một quốc gia", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))